# Serialización del mejor modelo y MLflow Model Registry

En este notebook serializamos el pipeline completo del mejor modelo
(**Exp 0: LogisticRegression + TF-IDF**, F1-macro test: 0.9530)
y lo registramos en el MLflow Model Registry.

Pasos:
1. Carga del mejor modelo
2. Serialización completa del pipeline con naming convention claro
3. Loguear el modelo en MLflow con `mlflow.sklearn.log_model`
4. Registrar en el Model Registry y transitar a stage `Production`
5. Verificación del registro

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Carga del mejor modelo

In [ ]:
import joblib
import pandas as pd

# Mejor modelo: Exp 0 (LogReg + TF-IDF baseline)
modelo = joblib.load("model/modelo_baseline.joblib")
tfidf  = joblib.load("model/tfidf_vectorizer.joblib")

print(f"Modelo cargado: {type(modelo).__name__}")
print(f"Vocabulario TF-IDF: {len(tfidf.vocabulary_)} términos")
print(f"Clases: {sorted(modelo.classes_.tolist())}")
print(f"Parámetros del modelo: {modelo.get_params()}")

## 2. Serialización completa del pipeline

In [ ]:
from functions import guardar_pipeline_completo

path_modelo, path_tfidf, path_meta = guardar_pipeline_completo(
    modelo=modelo,
    tfidf=tfidf,
    label_encoder=None,  # LogReg no necesita LabelEncoder
    metadata={
        "experimento":     "0",
        "nombre":          "exp0_logreg_tfidf_baseline",
        "test_f1_macro":   0.9530,
        "test_accuracy":   0.9556,
        "test_roc_auc":    0.9968,
        "features":        "tfidf_bigramas",
        "tfidf_n_terms":   len(tfidf.vocabulary_),
        "criterio":        "Mayor F1-macro en test entre los 3 experimentos",
    },
    output_dir="model",
)

## 3. Loguear el modelo en MLflow con sklearn.log_model

Usamos `mlflow.sklearn.log_model` con `registered_model_name` para que
MLflow cree automáticamente la primera versión en el Model Registry.

In [ ]:
import mlflow
import mlflow.sklearn
from functions import configure_mlflow, MLFLOW_EXPERIMENT

REGISTERED_NAME = "clasificador_riesgo_ia"

configure_mlflow()
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name="mejor_modelo_produccion") as run:
    run_id = run.info.run_id

    # Parámetros
    mlflow.log_params({
        "model_type":         "LogisticRegression",
        "tfidf_max_features": 5000,
        "tfidf_ngram_range":  "(1, 2)",
        "tfidf_sublinear_tf": True,
        "features":           "tfidf_bigramas",
    })

    # Métricas del test
    mlflow.log_metrics({
        "test_f1_macro":  0.9530,
        "test_accuracy":  0.9556,
        "test_roc_auc":   0.9968,
    })

    # Tags
    mlflow.set_tags({
        "best_model":     "true",
        "experimento":    "0",
        "criterio":       "F1-macro test",
    })

    # Artefactos joblib
    mlflow.log_artifact(path_modelo)
    mlflow.log_artifact(path_tfidf)
    mlflow.log_artifact(path_meta)

    # Loguear el modelo sklearn (crea la entrada en el Registry)
    mlflow.sklearn.log_model(
        sk_model=modelo,
        artifact_path="modelo",
        registered_model_name=REGISTERED_NAME,
    )

print(f"Run completado. Run ID: {run_id}")

## 4. Transitar a stage Production en el Model Registry

In [ ]:
from functions import registrar_modelo_en_registry

model_version = registrar_modelo_en_registry(
    run_id=run_id,
    artifact_path="modelo",
    registered_name=REGISTERED_NAME,
    stage="Production",
)

## 5. Verificación del registro

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Listar todas las versiones del modelo
versiones = client.search_model_versions(f"name='{REGISTERED_NAME}'")
print(f"Versiones registradas de '{REGISTERED_NAME}':")
for v in versiones:
    print(f"  v{v.version} | stage={v.current_stage} | run_id={v.run_id[:8]}...")

# Verificar que hay una versión en Production
prod_versions = [v for v in versiones if v.current_stage == "Production"]
if prod_versions:
    print(f"\n✓ Modelo en Production: v{prod_versions[0].version}")
else:
    print("\n⚠ No hay versiones en Production")

## Resumen final

| Artefacto | Ruta |
|-----------|------|
| Modelo serializado | `model/mejor_modelo.joblib` |
| Vectorizador serializado | `model/mejor_modelo_tfidf.joblib` |
| Metadata | `model/model_metadata.json` |
| MLflow Registry | `clasificador_riesgo_ia` (stage: Production) |

In [ ]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
# (El logging principal ya se hizo en la celda 3 con start_run.)
# Esta celda es un resumen de verificación final.
try:
    client = MlflowClient()
    versiones = client.search_model_versions(f"name='{REGISTERED_NAME}'")
    prod = [v for v in versiones if v.current_stage == "Production"]
    print(f"✓ '{REGISTERED_NAME}' — {len(versiones)} versión(es) registrada(s)")
    if prod:
        print(f"  En Production: v{prod[0].version} (run {prod[0].run_id[:8]}...)")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")